# No-Show Model — Experiment Tracker
Run each cell **top to bottom**. Never edit a cell you already ran — copy it into a new cell below to try changes.

## Cell 1 — Setup: Generate & Preprocess Data
Run this once. All experiments below share this same data.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

from src.data.generate import generate
from src.data.preprocess import preprocess
from src.utils.config import FEATURES, TARGET

raw_df = generate(n=3000, seed=42, save=False)
df     = preprocess(raw_df, save=False)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'No-show rate in test: {y_test.mean():.2%}')

## Cell 2 — Helper Function
A reusable function that prints metrics clearly. Run once.

In [ ]:
def evaluate(name, model, X_test, y_test, threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm  = confusion_matrix(y_test, y_pred)

    print(f'\n{"-"*50}')
    print(f'  EXPERIMENT: {name}')
    print(f'  Threshold used: {threshold}')
    print(f'{"-"*50}')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  ROC-AUC:       {auc:.4f}')
    print(f'  F1 (No-Show):  {f1:.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN={cm[0][0]}  FP={cm[0][1]}')
    print(f'    FN={cm[1][0]}  TP={cm[1][1]}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Show", "No-Show"])}')

    return {'name': name, 'accuracy': acc, 'f1': f1, 'roc_auc': auc, 'threshold': threshold}

results = []
print('Helper ready!')

## Experiment 1 — Baseline (class_weight=balanced, threshold=0.5)
This is what your original model was doing. Gives us a baseline to beat.

In [ ]:
model_v1 = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model_v1.fit(X_train, y_train)

r1 = evaluate('v1 - balanced, threshold=0.5', model_v1, X_test, y_test, threshold=0.5)
results.append(r1)

## Experiment 2 — Lower Threshold to 0.30
Same model as v1, but we lower the decision bar for predicting no-show.

In [ ]:
r2 = evaluate('v2 - balanced, threshold=0.30', model_v1, X_test, y_test, threshold=0.30)
results.append(r2)

## Experiment 3 — Stronger Class Weight {0:1, 1:4}
We manually tell the model: missing a no-show is 4x worse than missing a show.

In [ ]:
model_v3 = RandomForestClassifier(
    n_estimators=300,
    class_weight={0: 1, 1: 4},
    random_state=42,
    n_jobs=-1
)
model_v3.fit(X_train, y_train)

r3 = evaluate('v3 - weight {0:1,1:4}, threshold=0.30', model_v3, X_test, y_test, threshold=0.30)
results.append(r3)

## Experiment 4 — GridSearchCV Tuned (scoring=f1)
Automatically finds the best hyperparameters. Takes 1-2 minutes.

In [ ]:
param_grid = {
    'n_estimators':     [200, 300, 500],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight={0: 1, 1: 4},
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
model_v4 = grid_search.best_estimator_

r4 = evaluate('v4 - GridSearchCV, threshold=0.30', model_v4, X_test, y_test, threshold=0.30)
results.append(r4)

## Final — Compare All Experiments Side by Side
Run this after all experiments above are done.

In [ ]:
summary = pd.DataFrame(results)
summary = summary.set_index('name')
summary = summary.round(4)
print('\n=== EXPERIMENT SUMMARY ===')
print(summary.to_string())

best = summary['f1'].idxmax()
print(f'\nBest model by F1: {best}')